# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanzina-Aranya-Islam/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row in the warehouse daily performance table represents one content page for one client on one report date. For my Lane 2 project, the eventual decision unit will be a content page, and daily observations will be aggregated into page-level features at a clearly defined decision point.

**Time window:** For initial development, I will use the middle-panel month 2026-03. I will not use the `_sample` table for development because it represents the final month and could expose future information.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

**Features:** For the first feature frame, I will use GSC impressions, GSC clicks, GA4 sessions, average search position, and AI referral sessions. These are observable performance signals that describe search visibility, clicks, traffic, position, and referral context. They will only be used when they are available before the defined decision point.

**Label / proxy:** For comparison with the starter playground, I will use `trend_direction == "down"` as a proxy for declining performance when working with the starter dataset. For the warehouse work, I will eventually define a future-looking outcome rather than treating a current trend bucket as a true future label.

**Context:** `client_hash_id`, `content_hash_id`, and `report_date` will be used to identify, group, join, and define the observation period. The identifiers themselves will not be treated as meaningful predictive features.

**Excluded:** I will exclude identifying or sensitive information such as URLs, titles, client names, and private queries. I will also exclude product decision outputs such as `health_score`, `priority_score`, or `action_type` if encountered, because using existing decisions as features could create circular results.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [20]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from huggingface_hub import login

login(token=HF_TOKEN)

print("Hugging Face login successful")

Hugging Face login successful


In [22]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Number of files:", len(files))

for f in files[:30]:
    print(f)

Number of files: 24
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_da

In [23]:
#Query 1
import pandas as pd

path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

df_march = pd.read_parquet(
    path,
    storage_options={"token": HF_TOKEN}
)

print("Rows:", len(df_march))
print("Columns:", df_march.columns.tolist())

Rows: 9841378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [24]:
# Verify the grain of the March 2026 data

grain_check = (
    df_march
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
)

print("Total rows:", len(df_march))
print("Unique date-client-content combinations:", len(grain_check))
print("Maximum rows per combination:", grain_check.max())

Total rows: 9841378
Unique date-client-content combinations: 9841378
Maximum rows per combination: 1


In [25]:
# Query 2: Verify row count and date span

print("Row count:", len(df_march))
print("Minimum date:", df_march["report_date"].min())
print("Maximum date:", df_march["report_date"].max())

Row count: 9841378
Minimum date: 2026-03-01
Maximum date: 2026-03-31


In [26]:
# Query 3: Check data availability using IS TRUE

availability_check = df_march[
    df_march["gsc_data_available"].eq(True)
]

print("Rows with GSC data available:", len(availability_check))
print("Rows without GSC data available:", len(df_march) - len(availability_check))

Rows with GSC data available: 3611061
Rows without GSC data available: 6230317


## 4. Data limits

Data limits: The March 2026 slice is only one month of the warehouse, so it may not represent longer-term trends or seasonality. GSC and GA4 data are not available for every row, so missing measurements should not automatically be treated as zero performance. The warehouse also has unbalanced history across clients, and future-window labels would require strict separation between the feature and target periods to avoid leakage. Finally, the data is observational, so a high-ranked page should be treated as a review candidate, not as proof that a refresh will cause improvement.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.